# 04: Ma'lumotni tayyorlash va Pipeline
Maqsad: feature'larni bir xil masshtabga keltirish, matnli ustunni songa aylantirish
va hamma qadamlarni bitta zanjirga ulash.

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = sns.load_dataset("mpg").dropna()

def evaluate(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}")

num_features = ["weight", "horsepower", "displacement", "cylinders", "model_year"]

Mashq 1: Masshtab muammosini ko'rish

📖 Izoh: Feature'lar turli o'lchov birliklarida berilgan. Keling, ularning qiymatlari qanchalik farq qilishini ko'raylik.

In [3]:
X = df[num_features]
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.describe().loc[["mean", "std", "min", "max"]].round(1)

,weight,horsepower,displacement,cylinders,model_year
mean,2986.1,104.6,195.5,5.5,76.2
std,841.1,38.3,103.8,1.7,3.6
min,1613.0,46.0,70.0,3.0,70.0
max,5140.0,230.0,455.0,8.0,82.0


StandardScaler

📖 Izoh: StandardScaler har bir feature'ni shunday o'zgartiradiki, uning o'rtachasi 0, standart og'ishi 1 bo'ladi:

$$x_{yangi} = \frac{x - \text{o'rtacha}}{\text{standart og'ish}}$$

Shunda hamma feature'lar bir xil masshtabga keladi.

⚠️ Eng muhim qoida:

Train uchun → fit_transform(): o'rtacha va std'ni o'rganadi va qo'llaydi
Test uchun → faqat transform(): train'dan o'rganilgan qiymatlarni qo'llaydi

Nega? Test ma'lumoti modelning "kelajakdagi yangi ma'lumoti" vazifasini bajaradi. Agar scaler test'dan ham o'rganib qolsa, bu data leakage (ma'lumot sizib chiqishi) bo'ladi: imtihon javoblarini oldindan ko'rib qo'yishdek.

In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # o'rganadi + qo'llaydi
X_test_s  = scaler.transform(X_test)        # faqat qo'llaydi

pd.DataFrame(X_train_s, columns=num_features).describe().loc[["mean", "std"]].round(2)

,weight,horsepower,displacement,cylinders,model_year
mean,0.0,-0.0,0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0


In [5]:
model = LinearRegression()
model.fit(X_train_s, y_train)

evaluate(y_test, model.predict(X_test_s))
print(pd.Series(model.coef_, index=num_features).round(2).sort_values())

MAE=2.50  RMSE=3.25  R²=0.793
weight         -5.34
horsepower     -0.30
cylinders      -0.20
displacement    0.05
model_year      2.75
dtype: float64


Mashq 3: OneHotEncoder, ya'ni matnni songa aylantirish

📖 Izoh: origin ustunida "usa", "japan", "europe" kabi matnlar bor, model esa faqat sonlar bilan ishlaydi.

❌ Noto'g'ri yo'l: usa = 1, japan = 2, europe = 3. Bunda model "europe japan'dan kattaroq" yoki "japan usa'dan 2 barobar ko'p" deb o'ylab qoladi, ammo bu mantiqsiz.

✅ To'g'ri yo'l, One-Hot Encoding: har bir kategoriya uchun alohida 0/1 ustun yaratiladi.

origin	origin_europe	origin_japan	origin_usa
usa	    0	                 0	            1
japan	0	                 1	            0
europe	1	                 0	            0

In [6]:
sample = pd.DataFrame({"origin": ["usa", "japan", "europe", "japan"]})

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(sample)

print(encoder.get_feature_names_out())
print(encoded)

['origin_europe' 'origin_japan' 'origin_usa']
[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]]


Mashq 4: ColumnTransformer + Pipeline

📖 Izoh: Endi bizda ikki xil ustun bor:

Sonli ustunlar → StandardScaler
Matnli ustun → OneHotEncoder

ColumnTransformer har bir ustun guruhiga o'z ishlovini beradi.
Pipeline esa barcha qadamlarni bitta zanjirga ulaydi:

Xom ma'lumot → [Scaler + Encoder] → [LinearRegression] → Bashorat

🏭 Analogiya: zavoddagi konveyer. Xom ashyo bir tomondan kiradi va hamma bosqichlardan avtomatik o'tib, tayyor mahsulot bo'lib chiqadi.

Pipeline'ning foydalari:

fit() va predict() faqat bir marta chaqiriladi
Data leakage'ning oldini avtomatik oladi: train'da fit, test'da faqat transform qilinadi
Yangi ma'lumotni xom holida bersangiz bo'ladi, hammasini o'zi bajaradi

In [7]:
cat_features = ["origin"]

X = df[num_features + cat_features]
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", LinearRegression()),
])

pipe.fit(X_train, y_train)
evaluate(y_test, pipe.predict(X_test))

MAE=2.46  RMSE=3.26  R²=0.791


Mashq 5: Yangi mashina uchun bashorat va natijalarni tahlil qilish

📖 Izoh: Pipeline'ning eng qulay tomoni shundaki, yangi ma'lumotni xom ko'rinishida beramiz. Scaling va encoding'ni u o'zi bajaradi.

In [8]:
new_car = pd.DataFrame({
    "weight": [2500], "horsepower": [90], "displacement": [120],
    "cylinders": [4], "model_year": [80], "origin": ["japan"]
})
print("Bashorat:", pipe.predict(new_car))

# Pipeline ichidagi weight'larni ko'rish
names = pipe.named_steps["preprocess"].get_feature_names_out()
coefs = pipe.named_steps["model"].coef_
pd.Series(coefs, index=names).round(2)

Bashorat: [31.25612795]


num__weight          -5.28
num__horsepower      -0.97
num__displacement     1.96
num__cylinders       -0.59
num__model_year       2.88
cat__origin_europe    0.85
cat__origin_japan     1.19
cat__origin_usa      -2.03
dtype: float64